# HBV Model — Temperature & Evapotranspiration Analysis
**Region:** Mosel catchment  
**Temporal resolution:** Monthly (with seasonal breakdown)  
**Research question:** Does temperature have a significant effect on evapotranspiration, and is this relationship stronger than the previously tested temperature–sumax (γ) link?

---
### Workflow
1. Load & preprocess data
2. Exploratory analysis & seasonal decomposition
3. Linear regression (pooled & seasonal)
4. Partial correlation (controlling for seasonal cycle)
5. Nonlinear regression
6. Mutual information (nonlinear dependence check)
7. HBV model sensitivity analysis (temperature perturbation)
8. Bayesian / MCMC parameter estimation (ET scaling vs temperature)
9. Summary & uncertainty quantification

---
## 0. Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import mutual_info_regression
from sklearn.utils import resample
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# Optional: for MCMC (install with: pip install pymc)
# import pymc as pm
# import arviz as az

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False
})

SEASONS = {
    'DJF': [12, 1, 2],
    'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],
    'SON': [9, 10, 11]
}
SEASON_COLORS = {'DJF': '#5b8ec4', 'MAM': '#6abf69', 'JJA': '#e07b3a', 'SON': '#c17bca'}

print('Imports OK')

---
## 1. Load & preprocess data

**Expected input format:** A CSV or DataFrame with at least:
- `date` — monthly timestamps (YYYY-MM-DD or YYYY-MM)
- `temp` — mean monthly temperature (°C)
- `et` — monthly actual or potential ET (mm/month)
- `precip` — monthly precipitation (mm) [optional but recommended]
- `discharge` — monthly discharge (mm or m³/s) [optional]

> **Replace the synthetic data block below with your actual Mosel data.**

In [ ]:
# ============================================================
# --- OPTION A: Load your real data ---
# ============================================================
# df = pd.read_csv('mosel_monthly.csv', parse_dates=['date'])
# df = df.set_index('date').sort_index()

# ============================================================
# --- OPTION B: Synthetic Mosel-like data (placeholder) ---
# ============================================================
np.random.seed(42)
n_years = 40
dates = pd.date_range('1984-01-01', periods=n_years * 12, freq='MS')
months = dates.month

# Temperature: seasonal cycle + long-term warming trend + noise
temp_seasonal = 8 + 10 * np.sin((months - 1) / 12 * 2 * np.pi - np.pi / 2)
temp_trend = np.linspace(0, 1.5, len(dates))  # +1.5°C over 40 years
temp = temp_seasonal + temp_trend + np.random.normal(0, 1.2, len(dates))

# ET: driven by temperature + radiation proxy (seasonal) + noise
# Physical relationship: ET ~ a * max(T - T0, 0) — Hamon-like
T0 = 0.0   # threshold temperature
et_base = 3.5 * np.maximum(temp - T0, 0) ** 0.7  # nonlinear but temperature-driven
et = et_base + np.random.normal(0, 4, len(dates))
et = np.clip(et, 0, None)

# Precipitation
precip = 60 + 20 * np.sin((months + 2) / 12 * 2 * np.pi) + np.random.normal(0, 20, len(dates))
precip = np.clip(precip, 0, None)

df = pd.DataFrame({'temp': temp, 'et': et, 'precip': precip}, index=dates)
df.index.name = 'date'
df['month'] = df.index.month
df['year']  = df.index.year

# Assign season labels
def month_to_season(m):
    for s, months_list in SEASONS.items():
        if m in months_list:
            return s
df['season'] = df['month'].apply(month_to_season)

print(f'Dataset: {len(df)} monthly records ({df.index.year.min()}–{df.index.year.max()})')
df.describe().round(2)

---
## 2. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(df.index, df['temp'], color='#e07b3a', lw=0.8, alpha=0.7)
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Mosel — Monthly Climate Series', fontsize=13, fontweight='bold')

axes[1].plot(df.index, df['et'], color='#3a8abf', lw=0.8, alpha=0.7)
axes[1].set_ylabel('ET (mm/month)')

axes[2].bar(df.index, df['precip'], color='#5b8ec4', alpha=0.6, width=25)
axes[2].set_ylabel('Precipitation (mm/month)')

plt.tight_layout()
plt.show()

In [ ]:
# Monthly climatology — mean seasonal cycle
clim = df.groupby('month')[['temp', 'et']].mean()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.bar(clim.index, clim['et'], color='#3a8abf', alpha=0.65, label='ET')
ax2.plot(clim.index, clim['temp'], 'o-', color='#e07b3a', lw=2, ms=6, label='Temp')

ax1.set_xlabel('Month')
ax1.set_ylabel('ET (mm/month)', color='#3a8abf')
ax2.set_ylabel('Temperature (°C)', color='#e07b3a')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
plt.title('Mean seasonal cycle — ET and Temperature', fontweight='bold')
fig.tight_layout()
plt.show()
print('Note: Both variables peak in summer — we must control for this in regressions.')

---
## 3. Linear regression
### 3a. Pooled (all months) — naive baseline

In [ ]:
slope, intercept, r, p, se = stats.linregress(df['temp'], df['et'])

print('=== Pooled linear regression: ET ~ Temperature ===')
print(f'  Slope     : {slope:.3f} mm/month per °C')
print(f'  Intercept : {intercept:.3f}')
print(f'  R²        : {r**2:.3f}')
print(f'  p-value   : {p:.2e}')
print(f'  Std error : {se:.4f}')
print()
print('⚠️  High R² here is expected and likely driven by the shared seasonal cycle.')
print('   See Section 4 (partial correlation) for the deseasonalised signal.')

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
for s, sc in SEASON_COLORS.items():
    sub = df[df['season'] == s]
    ax.scatter(sub['temp'], sub['et'], color=sc, alpha=0.4, s=12, label=s)

x_line = np.linspace(df['temp'].min(), df['temp'].max(), 200)
ax.plot(x_line, intercept + slope * x_line, 'k--', lw=2, label=f'OLS (R²={r**2:.2f})')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('ET (mm/month)')
ax.set_title('Pooled ET ~ Temperature (coloured by season)', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3b. Per-season linear regression

In [ ]:
results_seasonal = {}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for i, (season, color) in enumerate(SEASON_COLORS.items()):
    sub = df[df['season'] == season].copy()
    sl, ic, r, p, se = stats.linregress(sub['temp'], sub['et'])
    results_seasonal[season] = dict(slope=sl, intercept=ic, r2=r**2, p=p, se=se, n=len(sub))

    ax = axes[i]
    ax.scatter(sub['temp'], sub['et'], color=color, alpha=0.5, s=15)
    x_l = np.linspace(sub['temp'].min(), sub['temp'].max(), 100)
    ax.plot(x_l, ic + sl * x_l, 'k-', lw=1.8)
    ax.set_title(f'{season}  |  R²={r**2:.3f}, p={p:.3f}, slope={sl:.2f}', fontsize=10)
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('ET (mm/month)')

plt.suptitle('Seasonal ET ~ Temperature regressions', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n=== Seasonal regression summary ===')
print(pd.DataFrame(results_seasonal).T.round(4))

### 3c. Bootstrap confidence intervals on seasonal slopes

In [ ]:
N_BOOT = 2000

boot_results = {}
for season in SEASONS:
    sub = df[df['season'] == season][['temp', 'et']].dropna().values
    slopes_boot = []
    for _ in range(N_BOOT):
        s = resample(sub)
        sl, *_ = stats.linregress(s[:, 0], s[:, 1])
        slopes_boot.append(sl)
    slopes_boot = np.array(slopes_boot)
    boot_results[season] = {
        'mean': slopes_boot.mean(),
        'ci_lo': np.percentile(slopes_boot, 2.5),
        'ci_hi': np.percentile(slopes_boot, 97.5),
        'slopes': slopes_boot
    }

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(SEASONS))
for i, (season, color) in enumerate(SEASON_COLORS.items()):
    br = boot_results[season]
    ax.bar(i, br['mean'], color=color, alpha=0.8, width=0.5)
    ax.errorbar(i, br['mean'],
                yerr=[[br['mean'] - br['ci_lo']], [br['ci_hi'] - br['mean']]],
                fmt='none', color='black', capsize=6, lw=2)

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x_pos)
ax.set_xticklabels(list(SEASON_COLORS.keys()))
ax.set_ylabel('Slope (mm/month per °C)')
ax.set_title('Bootstrap 95% CI on seasonal ET~Temperature slopes', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Bootstrap results (95% CI) ===')
for s, br in boot_results.items():
    print(f'  {s}: slope = {br["mean"]:.3f}  [{br["ci_lo"]:.3f}, {br["ci_hi"]:.3f}]')

---
## 4. Partial correlation — controlling for the seasonal cycle

This is the key analysis. We remove the mean seasonal cycle from both temperature and ET (i.e., compute anomalies), then regress the anomalies. This tests whether **interannual** temperature variability drives **interannual** ET variability — independent of the shared seasonal cycle.

In [ ]:
# Compute monthly climatology and anomalies
clim_mean = df.groupby('month')[['temp', 'et']].transform('mean')
df['temp_anom'] = df['temp'] - clim_mean['temp']
df['et_anom']   = df['et']   - clim_mean['et']

# Regression on anomalies
sl, ic, r, p, se = stats.linregress(df['temp_anom'], df['et_anom'])

print('=== Partial correlation (anomalies, seasonal cycle removed) ===')
print(f'  Slope     : {sl:.3f} mm/month per °C anomaly')
print(f'  R²        : {r**2:.3f}')
print(f'  p-value   : {p:.4f}')
print()
if r**2 > 0.05 and p < 0.05:
    print('✅ Significant interannual temperature–ET relationship after removing seasonal cycle.')
else:
    print('⚠️  Weak or non-significant after removing seasonal cycle. The pooled R² was likely inflated.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Anomaly scatter
for s, sc in SEASON_COLORS.items():
    sub = df[df['season'] == s]
    axes[0].scatter(sub['temp_anom'], sub['et_anom'], color=sc, alpha=0.4, s=12, label=s)
x_l = np.linspace(df['temp_anom'].min(), df['temp_anom'].max(), 200)
axes[0].plot(x_l, ic + sl * x_l, 'k--', lw=2)
axes[0].axhline(0, color='gray', lw=0.5)
axes[0].axvline(0, color='gray', lw=0.5)
axes[0].set_xlabel('Temperature anomaly (°C)')
axes[0].set_ylabel('ET anomaly (mm/month)')
axes[0].set_title(f'Anomaly regression  R²={r**2:.3f}, p={p:.4f}', fontweight='bold')
axes[0].legend(fontsize=8)

# Compare raw vs anomaly R²
sl_raw, _, r_raw, p_raw, _ = stats.linregress(df['temp'], df['et'])
bars = axes[1].bar(['Pooled (raw)', 'Anomaly\n(deseasonalised)'],
                   [r_raw**2, r**2],
                   color=['#e07b3a', '#3a8abf'], alpha=0.8, width=0.4)
for bar, pv in zip(bars, [p_raw, p]):
    label = f'p={pv:.3f}' if pv >= 0.001 else f'p<0.001'
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 label, ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1)
axes[1].set_title('R² comparison: raw vs deseasonalised', fontweight='bold')

plt.tight_layout()
plt.show()

### 4b. OLS with month dummies (statsmodels) — formal partial correlation

In [ ]:
# Formal regression controlling for month fixed effects
df_ols = df[['et', 'temp', 'month']].copy()
df_ols['month'] = df_ols['month'].astype('category')

model = smf.ols('et ~ temp + C(month)', data=df_ols).fit()

print('=== OLS with month fixed effects ===')
print(f'  temp coefficient : {model.params["temp"]:.4f}')
print(f'  p-value (temp)   : {model.pvalues["temp"]:.4f}')
print(f'  R² (full model)  : {model.rsquared:.4f}')
print(f'  AIC              : {model.aic:.2f}')
print()
print(model.summary().tables[1])

---
## 5. Nonlinear regression

In [ ]:
# --- Model functions ---
def hamon_like(T, a, T0, b):
    """ET ~ a * max(T - T0, 0)^b  — Hamon-inspired"""
    return a * np.maximum(T - T0, 0) ** b

def exponential_model(T, a, b):
    return a * np.exp(b * T)

def polynomial_model(T, a, b, c):
    return a * T**2 + b * T + c

T_data = df['temp'].values
ET_data = df['et'].values
sort_idx = np.argsort(T_data)
T_sorted = T_data[sort_idx]

fit_results = {}

# Hamon-like
try:
    popt, pcov = curve_fit(hamon_like, T_data, ET_data, p0=[3.5, 0, 0.7],
                           bounds=([0, -5, 0.1], [20, 10, 3]), maxfev=5000)
    et_pred = hamon_like(T_sorted, *popt)
    ss_res = np.sum((ET_data - hamon_like(T_data, *popt))**2)
    ss_tot = np.sum((ET_data - ET_data.mean())**2)
    fit_results['Hamon-like'] = dict(popt=popt, r2=1 - ss_res/ss_tot, pred=et_pred)
    print(f'Hamon-like: a={popt[0]:.3f}, T0={popt[1]:.2f}, b={popt[2]:.3f}, R²={1-ss_res/ss_tot:.3f}')
except Exception as e:
    print(f'Hamon fit failed: {e}')

# Polynomial degree 2
poly_model = make_pipeline(PolynomialFeatures(2), LinearRegression())
poly_model.fit(T_data.reshape(-1, 1), ET_data)
et_poly = poly_model.predict(T_sorted.reshape(-1, 1))
r2_poly = poly_model.score(T_data.reshape(-1, 1), ET_data)
fit_results['Polynomial (deg 2)'] = dict(r2=r2_poly, pred=et_poly)
print(f'Polynomial deg 2: R²={r2_poly:.3f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(T_data, ET_data, color='lightgray', s=10, alpha=0.6, label='Data')
colors_nl = ['#e07b3a', '#3a8abf', '#6abf69']
for (name, res), col in zip(fit_results.items(), colors_nl):
    ax.plot(T_sorted, res['pred'], color=col, lw=2, label=f'{name} (R²={res["r2"]:.3f})')

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('ET (mm/month)')
ax.set_title('Nonlinear regression models', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Mutual information — detecting nonlinear dependence

In [ ]:
# Compare mutual information (any dependence) vs linear correlation
features = ['temp', 'temp_anom', 'precip']
X = df[features].dropna().values
y = df.loc[df[features].dropna().index, 'et'].values

mi = mutual_info_regression(X, y, random_state=42)
corr_linear = [abs(np.corrcoef(X[:, i], y)[0, 1]) for i in range(X.shape[1])]

mi_df = pd.DataFrame({
    'feature': features,
    'mutual_info': mi,
    'abs_linear_corr': corr_linear
})

print('=== Mutual Information vs Linear Correlation ===')
print(mi_df.round(4).to_string(index=False))
print()
print('If MI >> linear corr for temp_anom, there is nonlinear signal beyond what linear regression captures.')

fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(len(features))
w = 0.3
ax.bar(x_pos - w/2, mi_df['mutual_info'], width=w, label='Mutual Information', color='#5b8ec4', alpha=0.8)
ax.bar(x_pos + w/2, mi_df['abs_linear_corr'], width=w, label='|Linear corr|', color='#e07b3a', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(features)
ax.set_title('Mutual Information vs Linear Correlation with ET', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. HBV model sensitivity analysis

We perturb the temperature input to the HBV model and observe how simulated ET responds.  
**Replace `hbv_et_from_temperature()` with your actual HBV model's ET routine.**

In [ ]:
# ============================================================
# STUB: Replace this with your actual HBV ET routine
# ============================================================
def hbv_compute_et(temp_series, lp=0.9, fc=200, sm=150, cet=0.15):
    """
    Simplified HBV ET routine (placeholder).
    
    In HBV, potential ET is often computed as:
        PET = (1 + cet * (T - T_mean)) * PET_mean
    and actual ET is:
        AET = PET * min(SM / (LP * FC), 1)
    
    Parameters
    ----------
    temp_series : array-like, monthly temperature (°C)
    lp          : LP parameter (fraction of FC above which AET = PET)
    fc          : field capacity (mm)
    sm          : soil moisture (mm, held constant here for sensitivity test)
    cet         : temperature correction factor for PET
    
    Replace with your real model call, e.g.:
        return my_hbv_model.run(temp=temp_series, precip=precip, ...)['et']
    """
    T_mean = 8.0  # long-term mean temperature
    PET_mean = 60.0  # long-term mean monthly PET (mm)
    PET = (1 + cet * (np.array(temp_series) - T_mean)) * PET_mean
    PET = np.maximum(PET, 0)
    AET = PET * min(sm / (lp * fc), 1.0)
    return AET

# Sensitivity: ΔT from -3 to +4°C
delta_T_range = np.arange(-3, 4.5, 0.5)
et_sensitivity = []

for dT in delta_T_range:
    perturbed_temp = df['temp'].values + dT
    et_sim = hbv_compute_et(perturbed_temp)
    et_sensitivity.append(et_sim.mean() if hasattr(et_sim, '__len__') else et_sim)

et_sensitivity = np.array(et_sensitivity)
et_baseline = et_sensitivity[np.argmin(np.abs(delta_T_range))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(delta_T_range, et_sensitivity, 'o-', color='#3a8abf', lw=2, ms=6)
ax.axvline(0, color='gray', ls='--', lw=0.8)
ax.axhline(et_baseline, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Temperature perturbation ΔT (°C)')
ax.set_ylabel('Mean simulated AET (mm/month)')
ax.set_title('HBV model: ET sensitivity to temperature perturbation', fontweight='bold')

# Annotate slope
sens_slope, *_ = stats.linregress(delta_T_range, et_sensitivity)
ax.annotate(f'Linear sensitivity: {sens_slope:.2f} mm/month/°C',
            xy=(1, et_baseline + sens_slope), fontsize=9,
            xytext=(0.05, 0.85), textcoords='axes fraction',
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))
plt.tight_layout()
plt.show()

print(f'HBV ET sensitivity: {sens_slope:.3f} mm/month per °C perturbation')

### 7b. Seasonal sensitivity breakdown

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for i, (season, color) in enumerate(SEASON_COLORS.items()):
    sub = df[df['season'] == season]
    et_sens_season = []
    for dT in delta_T_range:
        pt = sub['temp'].values + dT
        et_s = hbv_compute_et(pt)
        et_sens_season.append(et_s.mean() if hasattr(et_s, '__len__') else et_s)

    axes[i].plot(delta_T_range, et_sens_season, 'o-', color=color, lw=2, ms=5)
    axes[i].axvline(0, color='gray', ls='--', lw=0.7)
    sl_s, *_ = stats.linregress(delta_T_range, et_sens_season)
    axes[i].set_title(f'{season}  |  sensitivity: {sl_s:.2f} mm/month/°C', fontsize=10)
    axes[i].set_xlabel('ΔT (°C)')
    axes[i].set_ylabel('AET (mm/month)')

plt.suptitle('Seasonal HBV ET sensitivity to temperature', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Bayesian estimation — ET scaling parameter vs temperature

This mirrors your gamma/sumax analysis: treat the CET (temperature correction factor in HBV's ET routine) as a linear function of temperature and estimate the posterior.  

Two options are provided:  
- **Option A (lightweight):** Scipy-based grid search + likelihood surface — no extra dependencies  
- **Option B (full MCMC):** PyMC — uncomment if PyMC is installed

In [ ]:
# ============================================================
# Option A: Scipy likelihood-based estimation (no PyMC needed)
# ============================================================
from scipy.stats import norm

# Model: ET = cet_base + cet_slope * T + noise
# (linear model for ET scaling as function of T)
T_anom_obs = df['temp_anom'].values
ET_anom_obs = df['et_anom'].values

def log_likelihood(params, T, ET_obs):
    alpha, beta, log_sigma = params
    sigma = np.exp(log_sigma)
    ET_pred = alpha + beta * T
    return np.sum(norm.logpdf(ET_obs, loc=ET_pred, scale=sigma))

from scipy.optimize import minimize
result = minimize(
    lambda p: -log_likelihood(p, T_anom_obs, ET_anom_obs),
    x0=[0.0, 2.0, np.log(5)],
    method='Nelder-Mead'
)
alpha_mle, beta_mle, log_sigma_mle = result.x

print('=== MLE estimates (anomalies: ET_anom ~ alpha + beta * T_anom) ===')
print(f'  alpha (intercept): {alpha_mle:.4f}')
print(f'  beta  (slope/CET) : {beta_mle:.4f}')
print(f'  sigma (noise std) : {np.exp(log_sigma_mle):.4f}')

# Bootstrap posterior on beta
betas_boot = []
for _ in range(3000):
    idx = np.random.choice(len(T_anom_obs), len(T_anom_obs), replace=True)
    res = minimize(
        lambda p: -log_likelihood(p, T_anom_obs[idx], ET_anom_obs[idx]),
        x0=[alpha_mle, beta_mle, log_sigma_mle],
        method='Nelder-Mead'
    )
    betas_boot.append(res.x[1])

betas_boot = np.array(betas_boot)
ci_lo, ci_hi = np.percentile(betas_boot, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(betas_boot, bins=60, color='#3a8abf', alpha=0.75, edgecolor='white')
ax.axvline(beta_mle, color='black', lw=2, label=f'MLE = {beta_mle:.3f}')
ax.axvline(ci_lo, color='red', lw=1.5, ls='--', label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axvline(ci_hi, color='red', lw=1.5, ls='--')
ax.axvline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('β (ET anomaly sensitivity to temperature anomaly)')
ax.set_ylabel('Bootstrap count')
ax.set_title('Bootstrapped posterior of ET~Temperature slope (anomalies)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n95% CI on beta: [{ci_lo:.3f}, {ci_hi:.3f}]')
if ci_lo > 0:
    print('✅ CI does not include 0 — significant positive ET-temperature sensitivity.')
elif ci_hi < 0:
    print('⚠️  CI does not include 0 — significant NEGATIVE sensitivity (unexpected, check data).')
else:
    print('⚠️  CI includes 0 — cannot reject null hypothesis of no ET-temperature link (anomaly scale).')

In [ ]:
# ============================================================
# Option B: Full Bayesian MCMC with PyMC (uncomment to use)
# ============================================================
# import pymc as pm
# import arviz as az
#
# with pm.Model() as et_model:
#     # Priors
#     alpha = pm.Normal('alpha', mu=0, sigma=10)
#     beta  = pm.Normal('beta',  mu=0, sigma=5)   # ET sensitivity to T
#     sigma = pm.HalfNormal('sigma', sigma=10)
#
#     # Likelihood
#     mu = alpha + beta * T_anom_obs
#     ET_obs_pm = pm.Normal('ET_obs', mu=mu, sigma=sigma, observed=ET_anom_obs)
#
#     # Sample
#     trace = pm.sample(2000, tune=1000, target_accept=0.9,
#                       return_inferencedata=True, random_seed=42)
#
# az.plot_posterior(trace, var_names=['beta'], ref_val=0)
# plt.title('Posterior of ET temperature sensitivity (β)')
# plt.show()
#
# print(az.summary(trace, var_names=['alpha','beta','sigma']))
print('PyMC block is commented out. Uncomment above if PyMC is installed.')

---
## 9. Summary & comparison with gamma/sumax results

In [ ]:
print('=' * 60)
print('ANALYSIS SUMMARY')
print('=' * 60)

print('\n1. POOLED LINEAR REGRESSION (raw, seasonal cycle intact)')
sl0, _, r0, p0, _ = stats.linregress(df['temp'], df['et'])
print(f'   R² = {r0**2:.3f}, p = {p0:.2e}')
print('   ⚠️  Likely inflated by shared seasonal cycle.')

print('\n2. PARTIAL CORRELATION (anomalies, seasonal cycle removed)')
sl1, _, r1, p1, _ = stats.linregress(df['temp_anom'], df['et_anom'])
print(f'   R² = {r1**2:.3f}, p = {p1:.4f}')

print('\n3. SEASONAL SLOPES (bootstrapped 95% CI)')
for s, br in boot_results.items():
    sig = '✅' if br['ci_lo'] > 0 or br['ci_hi'] < 0 else '⚠️ '
    print(f'   {s}: {br["mean"]:.3f} [{br["ci_lo"]:.3f}, {br["ci_hi"]:.3f}]  {sig}')

print('\n4. HBV MODEL SENSITIVITY')
print(f'   Linear sensitivity: {sens_slope:.3f} mm/month per °C')

print('\n5. BAYESIAN / BOOTSTRAP MLE (anomalies)')
print(f'   β = {beta_mle:.3f}, 95% CI = [{ci_lo:.3f}, {ci_hi:.3f}]')
if ci_lo > 0:
    verdict = 'SIGNIFICANT — temperature anomalies predict ET anomalies.'
else:
    verdict = 'NOT SIGNIFICANT at anomaly scale — effect is largely seasonal.'
print(f'   Verdict: {verdict}')

print('\n6. COMPARISON TO GAMMA/SUMAX RESULT')
print('   Your previous result: temperature → γ/sumax: very small R², large error bars.')
if r1**2 > 0.05 and p1 < 0.05:
    print('   Current result:       temperature → ET: STRONGER signal detected.')
    print('   → Supports shifting research focus to ET as the temperature-sensitive component.')
else:
    print('   Current result:       temperature → ET (anomaly): similarly weak signal.')
    print('   → The relationship may be primarily seasonal, not interannual.')
    print('   → Consider: soil moisture, radiation, or humidity as confounders.')

print('\n' + '=' * 60)

---
## Next steps

1. **Replace synthetic data** with your real Mosel temperature, ET and HBV output.
2. **Plug in your real HBV model** in Section 7 — replace `hbv_compute_et()` with a call to your actual model.
3. **Run PyMC block** in Section 8 for full posterior distributions (comparable to your gamma MCMC).
4. Consider adding **humidity / radiation** as covariates in the OLS (Section 4b) to test whether temperature effect is mediated by VPD.
5. If the anomaly signal is weak, consider testing on **annual anomalies** instead of monthly (longer-term trends rather than interannual noise).